<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

### Prerequisito: cargar la base de datos

Antes de abrir esta notebook, importá el archivo `northwind_mysql.sql` en MySQL.

---

#### 🪟 Windows — MySQL Installer (instalación nativa)

**Opción A — MySQL Command Line Client** (recomendada, viene con MySQL Installer):

1. Abrí el menú Inicio → buscá **MySQL 8.0 Command Line Client** y ejecutalo
2. Te va a pedir la contraseña de root → ingresála
3. Ya estás dentro del shell de MySQL (vas a ver el prompt `mysql>`)
4. Ejecutá el siguiente comando (ajustá la ruta a donde tengas el archivo):
```sql
source C:/Users/TuUsuario/Desktop/northwind_mysql.sql
```
> 💡 Usá barras `/` en la ruta, no `\`. Ese es el error más común en Windows.

**Opción B — MySQL Workbench** (interfaz gráfica):

1. Abrí MySQL Workbench → conectate a tu servidor local
2. Menú **File → Open SQL Script** → seleccioná `northwind_mysql.sql`
3. Presioná el botón ⚡ (Execute) o `Ctrl+Shift+Enter`

**Opción C — Símbolo del sistema (cmd)**:

1. Abrí `cmd` como administrador
2. Navegá a la carpeta donde está el archivo:
```bat
cd C:\Users\TuUsuario\Desktop
```
3. Ejecutá (todo en una línea):
```bat
"C:\Program Files\MySQL\MySQL Server 8.0\bin\mysql.exe" -u root -p < northwind_mysql.sql
```

---

#### 🐧 Linux / macOS

```bash
mysql -u root -p < northwind_mysql.sql
```

---

#### ✅ Verificación (cualquier sistema)

Una vez importado, verificá que las tablas existen:
```sql
USE northwind;
SHOW TABLES;
```
Deberías ver 8 tablas: Categories, Customers, Employees, OrderDetails, Orders, Products, Shippers, Suppliers.

---
## PARTE 0 — Conexión a MySQL

Usamos `mysql-connector-python` para conectarnos y `pandas` para leer los resultados como DataFrames.

```bash
pip install mysql-connector-python pandas
```

In [1]:
import mysql.connector
import pandas as pd

CONN_PARAMS = {
    'host'    : 'localhost',
    'user'    : 'root',       # ← cambiá si usás otro usuario
    'password': 'v08983523',           # ← tu contraseña de MySQL (string vacío si no tiene)
    'database': 'northwind',
    'charset' : 'utf8mb4',
}

# Abrimos la conexión una sola vez para toda la notebook.
# Al terminar de trabajar, ejecutá la última celda para cerrarla.
conn = mysql.connector.connect(**CONN_PARAMS)
print('✅ Conexión abierta')


def query(sql, params=None):
    """
    Ejecuta una consulta SQL y devuelve un DataFrame.

    Parámetros:
        sql    : string con la consulta SQL
        params : tupla o dict de parámetros (para consultas parametrizadas)

    Ejemplo:
        query("SELECT * FROM Customers WHERE Country = %s", ('Germany',))
    """
    return pd.read_sql(sql, conn, params=params)


✅ Conexión abierta


In [2]:
# Resumen rápido: filas por tabla
resumen = query("""
    SELECT 'Categories'   AS Tabla, COUNT(*) AS Filas FROM Categories  UNION ALL
    SELECT 'Suppliers',              COUNT(*)          FROM Suppliers   UNION ALL
    SELECT 'Products',               COUNT(*)          FROM Products    UNION ALL
    SELECT 'Customers',              COUNT(*)          FROM Customers   UNION ALL
    SELECT 'Employees',              COUNT(*)          FROM Employees   UNION ALL
    SELECT 'Shippers',               COUNT(*)          FROM Shippers    UNION ALL
    SELECT 'Orders',                 COUNT(*)          FROM Orders      UNION ALL
    SELECT 'OrderDetails',           COUNT(*)          FROM OrderDetails
""")
print(resumen.to_string(index=False))

       Tabla  Filas
  Categories      8
   Suppliers     10
    Products     50
   Customers     40
   Employees      9
    Shippers      3
      Orders     94
OrderDetails    220


C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


---
# PARTE 1 — Nivel Básico
## SELECT, WHERE, ORDER BY, LIMIT

Todas las consultas siguen la misma estructura:

```sql
SELECT  columnas
FROM    tabla
WHERE   condición
ORDER BY columna ASC|DESC
LIMIT   n;
```

### 1.1 — Ver todas las categorías de productos

In [3]:
query("SELECT * FROM Categories")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CategoryID,CategoryName,Description
0,1,Beverages,"Soft drinks, coffees, teas, beers and ales"
1,2,Condiments,"Sweet and savory sauces, relishes, spreads and..."
2,3,Confections,"Desserts, candies and sweet breads"
3,4,Dairy Products,Cheeses
4,5,Grains/Cereals,"Breads, crackers, pasta and cereal"
5,6,Meat/Poultry,Prepared meats
6,7,Produce,Dried fruit and bean curd
7,8,Seafood,Seaweed and fish


### 1.2 — Productos con su precio y stock

In [4]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock,
           Discontinued
    FROM   Products
    ORDER BY UnitPrice DESC
    LIMIT 10
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,UnitPrice,UnitsInStock,Discontinued
0,C├┤te de Blaye,263.50,17,0
1,Th├╝ringer Rostbratwurst,123.79,0,1
2,Mishi Kobe Niku,97.00,29,1
3,Sir Rodney's Marmalade,81.00,40,0
4,Carnarvon Tigers,62.50,42,0
5,Ipoh Coffee,46.00,17,0
6,R├Âsti,45.60,26,0
7,R├Âssle Sauerkraut,45.60,26,1
8,Schoggi Schokolade,43.90,49,0
9,Northwoods Cranberry Sauce,40.00,6,0


### 1.3 — Filtrar por condición: productos caros con stock disponible

In [5]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock
    FROM   Products
    WHERE  UnitPrice > 50
      AND  UnitsInStock > 0
      AND  Discontinued = 0
    ORDER BY UnitPrice DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,UnitPrice,UnitsInStock
0,C├┤te de Blaye,263.5,17
1,Sir Rodney's Marmalade,81.0,40
2,Carnarvon Tigers,62.5,42


### 1.4 — Clientes de un país específico

> 💡 Usamos **consultas parametrizadas** con `%s` para evitar SQL injection.

In [6]:
pais = 'Germany'

query("""
    SELECT CustomerID,
           CompanyName,
           ContactName,
           City
    FROM   Customers
    WHERE  Country = %s
    ORDER BY CompanyName
""", (pais,))

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CustomerID,CompanyName,ContactName,City
0,ALFKI,Alfreds Futterkiste,Maria Anders,Berlin
1,BLAUS,Blauer See Delikatessen,Hanna Moos,Mannheim
2,DRACD,Drachenblut Delikatessen,Sven Ottlieb,Aachen
3,FRANK,Frankenversand,Peter Franken,M├╝nchen
4,KOENE,K├Âniglich Essen,Philip Cramer,Brandenburg


### 1.5 — Búsqueda con LIKE: productos que contienen una palabra

In [7]:
query("""
    SELECT ProductName, UnitPrice, UnitsInStock
    FROM   Products
    WHERE  ProductName LIKE '%Cheese%'
       OR  ProductName LIKE '%Queso%'
    ORDER BY ProductName
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,UnitPrice,UnitsInStock
0,Queso Cabrales,21.0,22
1,Queso Manchego La Pastora,38.0,86


### 1.6 — Columnas calculadas: total en stock

Podemos crear columnas calculadas directamente en SQL.

In [8]:
query("""
    SELECT ProductName,
           UnitPrice,
           UnitsInStock,
           ROUND(UnitPrice * UnitsInStock, 2) AS ValorEnStock
    FROM   Products
    WHERE  Discontinued = 0
    ORDER BY ValorEnStock DESC
    LIMIT 10
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,UnitPrice,UnitsInStock,ValorEnStock
0,C├┤te de Blaye,263.5,17,4479.5
1,Queso Manchego La Pastora,38.0,86,3268.0
2,Sir Rodney's Marmalade,81.0,40,3240.0
3,Grandma's Boysenberry,25.0,120,3000.0
4,Carnarvon Tigers,62.5,42,2625.0
5,Boston Crab Meat,18.4,123,2263.2
6,Schoggi Schokolade,43.9,49,2151.1
7,Inlagd Sill,19.0,112,2128.0
8,Sasquatch Ale,14.0,111,1554.0
9,R├Âd Kaviar,15.0,101,1515.0


### 1.7 — Pedidos en un rango de fechas

In [9]:
query("""
    SELECT OrderID,
           CustomerID,
           OrderDate,
           ShipCountry,
           Freight
    FROM   Orders
    WHERE  OrderDate BETWEEN '2023-10-01' AND '2023-10-31'
    ORDER BY OrderDate
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,OrderID,CustomerID,OrderDate,ShipCountry,Freight
0,66,ERNSH,2023-10-01,Austria,23.94
1,67,FOLKO,2023-10-02,Sweden,66.29
2,68,BOLID,2023-10-03,Spain,23.29
3,69,AROUT,2023-10-04,UK,13.34
4,70,ERNSH,2023-10-07,Austria,60.26
5,71,RATTC,2023-10-08,USA,2.10
6,72,FAMIA,2023-10-09,Brazil,37.52
7,73,GOURL,2023-10-10,Brazil,51.00
8,74,HUNGC,2023-10-11,USA,7.56
9,75,ERNSH,2023-10-14,Austria,48.77


### 1.8 — Productos descontinuados vs activos

In [10]:
query("""
    SELECT
        CASE Discontinued
            WHEN 0 THEN 'Activo'
            WHEN 1 THEN 'Descontinuado'
        END AS Estado,
        COUNT(*) AS Cantidad
    FROM Products
    GROUP BY Discontinued
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,Estado,Cantidad
0,Activo,42
1,Descontinuado,8


---
## ✏️ Ejercicios — Parte 1


**Ejercicio 1.** Listá todos los empleados ordenados por fecha de contratación (más antiguos primero). Mostrá nombre, apellido, cargo y fecha.

In [11]:
# Ejercicio 1


**Ejercicio 2.** ¿Cuántos clientes hay en cada país? Mostrá los 10 países con más clientes.


In [12]:
# Ejercicio 2


**Ejercicio 3.** Encontrá los productos cuyo stock está por debajo del nivel de reposición (`ReorderLevel`). Mostrá nombre, stock actual y nivel de reposición, ordenados por la diferencia.


In [13]:
# Ejercicio 3


**Ejercicio 4.** Listá los pedidos enviados a Brasil. ¿Cuál fue el flete más caro?


In [14]:
# Ejercicio 4


**Ejercicio 5.** Mostrá los pedidos del mes de noviembre 2023, incluyendo una columna calculada que indique cuántos días tardó en enviarse (`ShippedDate - OrderDate`).

In [15]:
# Ejercicio 5


---
# PARTE 2 — Nivel Intermedio
## JOIN, GROUP BY, Subconsultas

### Repaso visual de JOINs

```
INNER JOIN          LEFT JOIN          RIGHT JOIN
  A ∩ B               A ∪ (A∩B)          B ∪ (A∩B)
  ██████               ██████████         ██████████
 ███████              ████████████       ████████████
 solo lo que         todo A +            todo B +
 coincide en         lo que coincide     lo que coincide
 ambas tablas        en B                en A
```

### 2.1 — INNER JOIN: productos con su categoría y proveedor

In [16]:
query("""
    SELECT p.ProductName,
           c.CategoryName,
           s.CompanyName    AS Supplier,
           s.Country        AS SupplierCountry,
           p.UnitPrice
    FROM   Products   p
    JOIN   Categories c ON p.CategoryID  = c.CategoryID
    JOIN   Suppliers  s ON p.SupplierID  = s.SupplierID
    WHERE  p.Discontinued = 0
    ORDER BY c.CategoryName, p.UnitPrice DESC
    LIMIT 15
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,CategoryName,Supplier,SupplierCountry,UnitPrice
0,C├┤te de Blaye,Beverages,Exotic Liquids,UK,263.50
1,Ipoh Coffee,Beverages,Mayumis,Japan,46.00
2,Chang,Beverages,Exotic Liquids,UK,19.00
3,Chai,Beverages,Exotic Liquids,UK,18.00
4,Steeleye Stout,Beverages,New Orleans Cajun Delights,USA,18.00
5,Chartreuse verte,Beverages,Exotic Liquids,UK,18.00
6,Lakkalik├Â├Âri,Beverages,Mayumis,Japan,18.00
7,Sasquatch Ale,Beverages,New Orleans Cajun Delights,USA,14.00
8,Rh├Ânbr├ñu Klosterbier,Beverages,PB Kn├ñckebr├Âd AB,Sweden,7.75
9,Northwoods Cranberry Sauce,Condiments,Grandma Kellys Homestead,USA,40.00


### 2.2 — JOIN con 4 tablas: detalle completo de pedidos

In [17]:
query("""
    SELECT o.OrderID,
           o.OrderDate,
           c.CompanyName                                  AS Cliente,
           CONCAT(e.FirstName, ' ', e.LastName)           AS Empleado,
           p.ProductName,
           od.Quantity                                    AS Cantidad,
           od.UnitPrice                                   AS PrecioUnit,
           od.Discount                                    AS Descuento,
           ROUND(od.Quantity * od.UnitPrice * (1 - od.Discount), 2) AS Total
    FROM   Orders      o
    JOIN   Customers   c  ON o.CustomerID  = c.CustomerID
    JOIN   Employees   e  ON o.EmployeeID  = e.EmployeeID
    JOIN   OrderDetails od ON o.OrderID    = od.OrderID
    JOIN   Products    p  ON od.ProductID  = p.ProductID
    ORDER BY o.OrderDate DESC
    LIMIT 15
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,OrderID,OrderDate,Cliente,Empleado,ProductName,Cantidad,PrecioUnit,Descuento,Total
0,94,2023-11-08,Berglunds snabbk├Âp,Anne Dodsworth,Singaporean Hokkien,20,7.7,0.00,154.0
1,93,2023-11-07,GROSELLA-Restaurante,Andrew Fuller,Gravad lax,10,19.0,0.00,190.0
2,92,2023-11-06,Alfreds Futterkiste,Margaret Peacock,Queso Cabrales,24,14.0,0.00,336.0
3,92,2023-11-06,Alfreds Futterkiste,Margaret Peacock,Tunnbr├Âd,12,16.8,0.00,201.6
4,91,2023-11-05,Hungry Owl All-Night Grocers,Andrew Fuller,Alice Mutton,15,39.0,0.00,585.0
5,91,2023-11-05,Hungry Owl All-Night Grocers,Andrew Fuller,Ipoh Coffee,30,9.8,0.00,294.0
6,90,2023-11-04,Folk och f├ñ HB,Nancy Davolio,Th├╝ringer Rostbratwurst,10,36.4,0.05,345.8
7,90,2023-11-04,Folk och f├ñ HB,Nancy Davolio,Longlife Tofu,20,15.5,0.05,294.5
8,89,2023-11-01,Hanari Carnes,Michael Suyama,Jack's New England Clam,15,18.4,0.00,276.0
9,89,2023-11-01,Hanari Carnes,Michael Suyama,Singaporean Hokkien,25,7.7,0.00,192.5


### 2.3 — GROUP BY: ventas totales por país

`GROUP BY` agrupa filas y permite aplicar **funciones de agregación**: `SUM`, `COUNT`, `AVG`, `MAX`, `MIN`.

In [18]:
query("""
    SELECT  c.Country,
            COUNT(DISTINCT o.OrderID)                                AS TotalPedidos,
            COUNT(DISTINCT o.CustomerID)                             AS Clientes,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS RevenueTotal,
            ROUND(AVG(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS TicketPromedio
    FROM    Orders      o
    JOIN    Customers   c  ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od ON o.OrderID    = od.OrderID
    GROUP BY c.Country
    ORDER BY RevenueTotal DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,Country,TotalPedidos,Clientes,RevenueTotal,TicketPromedio
0,Germany,15,4,11408.95,325.97
1,Brazil,11,3,10217.40,425.72
2,USA,8,2,8298.45,488.14
3,Sweden,11,2,7798.50,339.07
4,France,14,3,6719.85,216.77
5,Canada,4,1,5327.87,591.99
6,Austria,9,1,5251.08,276.37
7,Ireland,4,1,3946.20,493.28
8,Spain,5,2,3511.00,319.18
9,Venezuela,5,2,2765.75,251.43


### 2.4 — HAVING: filtrar grupos

`HAVING` es como `WHERE` pero **después de agrupar**. Solo permite condiciones sobre columnas agregadas.

In [19]:
query("""
    SELECT  c.Country,
            COUNT(DISTINCT o.OrderID) AS TotalPedidos,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS Revenue
    FROM    Orders       o
    JOIN    Customers    c  ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od ON o.OrderID     = od.OrderID
    GROUP BY c.Country
    HAVING  COUNT(DISTINCT o.OrderID) >= 5    -- solo países con 5+ pedidos
    ORDER BY Revenue DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,Country,TotalPedidos,Revenue
0,Germany,15,11408.95
1,Brazil,11,10217.40
2,USA,8,8298.45
3,Sweden,11,7798.50
4,France,14,6719.85
5,Austria,9,5251.08
6,Spain,5,3511.00
7,Venezuela,5,2765.75


### 2.5 — Ranking de empleados por ventas

In [20]:
query("""
    SELECT  CONCAT(e.FirstName, ' ', e.LastName)   AS Empleado,
            e.Title,
            COUNT(DISTINCT o.OrderID)              AS TotalPedidos,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS VentasTotal
    FROM    Employees    e
    JOIN    Orders       o  ON e.EmployeeID  = o.EmployeeID
    JOIN    OrderDetails od ON o.OrderID     = od.OrderID
    GROUP BY e.EmployeeID, e.FirstName, e.LastName, e.Title
    ORDER BY VentasTotal DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,Empleado,Title,TotalPedidos,VentasTotal
0,Margaret Peacock,Sales Representative,19,14705.40
1,Janet Leverling,Sales Representative,16,10790.70
2,Laura Callahan,Inside Sales Coordinator,9,10351.00
3,Robert King,Sales Representative,9,7349.10
4,Nancy Davolio,Sales Representative,10,7300.50
5,Michael Suyama,Sales Representative,11,6752.80
6,Andrew Fuller,Vice President of Sales,9,5918.40
7,Anne Dodsworth,Sales Representative,6,5085.50
8,Steven Buchanan,Sales Manager,5,2967.49


### 2.6 — Ventas por categoría y mes

In [21]:
query("""
    SELECT  DATE_FORMAT(o.OrderDate, '%Y-%m') AS Mes,
            cat.CategoryName,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS Revenue
    FROM    Orders       o
    JOIN    OrderDetails od  ON o.OrderID    = od.OrderID
    JOIN    Products     p   ON od.ProductID = p.ProductID
    JOIN    Categories   cat ON p.CategoryID = cat.CategoryID
    GROUP BY Mes, cat.CategoryName
    ORDER BY Mes, Revenue DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,Mes,CategoryName,Revenue
0,2023-07,Beverages,3630.38
1,2023-07,Confections,2609.90
2,2023-07,Meat/Poultry,2395.25
3,2023-07,Seafood,1914.00
4,2023-07,Condiments,865.00
5,2023-07,Dairy Products,672.00
6,2023-07,Grains/Cereals,408.80
7,2023-07,Produce,390.60
8,2023-08,Seafood,4255.60
9,2023-08,Beverages,3774.80


### 2.7 — LEFT JOIN: clientes que nunca compraron

In [22]:
query("""
    SELECT  c.CustomerID,
            c.CompanyName,
            c.Country,
            COUNT(o.OrderID) AS TotalPedidos
    FROM    Customers c
    LEFT JOIN Orders  o ON c.CustomerID = o.CustomerID
    GROUP BY c.CustomerID, c.CompanyName, c.Country
    HAVING  TotalPedidos = 0
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CustomerID,CompanyName,Country,TotalPedidos
0,ANATR,Ana Trujillo Emparedados,Mexico,0
1,ANTON,Antonio Moreno Taquer├¡a,Mexico,0
2,BSBEV,B's Beverages,UK,0
3,COMMI,Com├®rcio Mineiro,Brazil,0
4,CONSH,Consolidated Holdings,UK,0
5,DUMON,Du monde entier,France,0
6,EASTC,Eastern Connection,UK,0
7,FISSA,FISSA Fabrica Inter.,Spain,0
8,FRANR,France restauration,France,0
9,FRANS,Franchi S.p.A.,Italy,0


### 2.8 — Subconsulta: productos con precio mayor al promedio de su categoría

In [23]:
query("""
    SELECT  p.ProductName,
            cat.CategoryName,
            p.UnitPrice,
            ROUND(prom.PromedioCat, 2) AS PromedioDeLaCategoria
    FROM    Products    p
    JOIN    Categories  cat ON p.CategoryID = cat.CategoryID
    JOIN (
        -- Subconsulta: precio promedio por categoría
        SELECT  CategoryID,
                AVG(UnitPrice) AS PromedioCat
        FROM    Products
        WHERE   Discontinued = 0
        GROUP BY CategoryID
    ) prom ON p.CategoryID = prom.CategoryID
    WHERE   p.UnitPrice > prom.PromedioCat
      AND   p.Discontinued = 0
    ORDER BY cat.CategoryName, p.UnitPrice DESC
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ProductName,CategoryName,UnitPrice,PromedioDeLaCategoria
0,C├┤te de Blaye,Beverages,263.50,46.92
1,Northwoods Cranberry Sauce,Condiments,40.00,22.50
2,Grandma's Boysenberry,Condiments,25.00,22.50
3,Sir Rodney's Marmalade,Confections,81.00,29.54
4,Schoggi Schokolade,Confections,43.90,29.54
5,Gumb├ñr Gummib├ñrchen,Confections,31.23,29.54
6,Queso Manchego La Pastora,Dairy Products,38.00,23.19
7,Mozzarella di Giovanni,Dairy Products,34.80,23.19
8,Mascarpone Fabioli,Dairy Products,32.00,23.19
9,R├Âsti,Grains/Cereals,45.60,27.30


### 2.9 — Subconsulta correlacionada: el pedido más grande de cada cliente

In [24]:
query("""
    SELECT  c.CompanyName,
            c.Country,
            o.OrderID,
            o.OrderDate,
            ROUND(SUM(od.Quantity * od.UnitPrice * (1 - od.Discount)), 2) AS TotalPedido
    FROM    Orders       o
    JOIN    Customers    c   ON o.CustomerID  = c.CustomerID
    JOIN    OrderDetails od  ON o.OrderID     = od.OrderID
    GROUP BY o.OrderID, c.CompanyName, c.Country, o.OrderDate
    HAVING  TotalPedido = (
        -- Subconsulta: máximo revenue para ese cliente
        SELECT MAX(sub_total)
        FROM (
            SELECT  o2.CustomerID,
                    SUM(od2.Quantity * od2.UnitPrice * (1 - od2.Discount)) AS sub_total
            FROM    Orders       o2
            JOIN    OrderDetails od2 ON o2.OrderID = od2.OrderID
            WHERE   o2.CustomerID = o.CustomerID
            GROUP BY o2.OrderID
        ) t
    )
    ORDER BY TotalPedido DESC
    LIMIT 10
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CompanyName,Country,OrderID,OrderDate,TotalPedido
0,Hungry Coyote Import Store,USA,42,2023-08-28,3186.4
1,Hanari Carnes,Brazil,14,2023-07-19,2577.0
2,Alfreds Futterkiste,Germany,2,2023-07-05,1863.4
3,Hungry Owl All-Night Grocers,Ireland,33,2023-08-15,1536.0
4,Great Lakes Food Market,USA,44,2023-08-30,1440.0
5,B├│lido Comidas preparadas,Spain,35,2023-08-19,1349.0
6,Ernst Handel,Austria,16,2023-07-23,1070.5
7,Frankenversand,Germany,77,2023-10-16,1064.4
8,Around the Horn,UK,32,2023-08-14,1056.0
9,Blauer See Delikatessen,Germany,19,2023-07-26,945.0


### 2.10 — Usar la vista `vw_orden_detalle`

El script SQL creó una vista que ya resuelve todos los JOINs. Las vistas simplifican consultas complejas.

In [25]:
# Vista completa
query("SELECT * FROM vw_orden_detalle LIMIT 10")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,OrderID,OrderDate,ShipCountry,Customer,CustomerCountry,FirstName,LastName,ProductName,CategoryName,Quantity,UnitPrice,Discount,LineTotal
0,11,2023-07-17,Austria,Ernst Handel,Austria,Nancy,Davolio,Boston Crab Meat,Seafood,3,18.0,0.0,54.0
1,21,2023-07-30,Canada,Bottom-Dollar Markets,Canada,Nancy,Davolio,Chef Anton's Gumbo Mix,Condiments,20,17.0,0.0,340.0
2,34,2023-08-16,Belgium,Bottom-Dollar Markets,Canada,Nancy,Davolio,Chang,Beverages,50,15.2,0.0,760.0
3,34,2023-08-16,Belgium,Bottom-Dollar Markets,Canada,Nancy,Davolio,Queso Cabrales,Dairy Products,30,14.0,0.0,420.0
4,51,2023-09-10,Austria,Ernst Handel,Austria,Nancy,Davolio,Nord-Ost Matjeshering,Seafood,6,99.0,0.0,594.0
5,51,2023-09-10,Austria,Ernst Handel,Austria,Nancy,Davolio,Gorgonzola Telino,Dairy Products,18,20.7,0.0,372.6
6,57,2023-09-18,Germany,Frankenversand,Germany,Nancy,Davolio,Th├╝ringer Rostbratwurst,Meat/Poultry,14,36.4,0.0,509.6
7,57,2023-09-18,Germany,Frankenversand,Germany,Nancy,Davolio,Nord-Ost Matjeshering,Seafood,3,99.0,0.0,297.0
8,64,2023-09-27,Brazil,Hanari Carnes,Brazil,Nancy,Davolio,Queso Cabrales,Dairy Products,20,14.0,0.0,280.0
9,64,2023-09-27,Brazil,Hanari Carnes,Brazil,Nancy,Davolio,Gorgonzola Telino,Dairy Products,15,20.7,0.0,310.5


In [26]:
# Ahora es muy simple consultar sobre la vista
query("""
    SELECT  CustomerCountry,
            CategoryName,
            ROUND(SUM(LineTotal), 2) AS Revenue
    FROM    vw_orden_detalle
    GROUP BY CustomerCountry, CategoryName
    ORDER BY Revenue DESC
    LIMIT 15
""")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,CustomerCountry,CategoryName,Revenue
0,USA,Seafood,3068.00
1,Brazil,Confections,2430.00
2,USA,Beverages,2017.80
3,Sweden,Seafood,1963.60
4,France,Seafood,1679.20
5,USA,Meat/Poultry,1532.05
6,Brazil,Beverages,1495.00
7,Canada,Meat/Poultry,1482.00
8,Sweden,Meat/Poultry,1437.80
9,Austria,Seafood,1424.40


### 2.11 — Integración Python + SQL: análisis dinámico

In [27]:
# Traemos los datos y hacemos el análisis final en Pandas
df_ventas = query("""
    SELECT  DATE_FORMAT(OrderDate, '%Y-%m') AS Mes,
            ROUND(SUM(LineTotal), 2)        AS Revenue,
            COUNT(DISTINCT OrderID)         AS Pedidos
    FROM    vw_orden_detalle
    GROUP BY Mes
    ORDER BY Mes
""")

df_ventas['TicketPromedio'] = (df_ventas['Revenue'] / df_ventas['Pedidos']).round(2)
df_ventas['RevenuePct']     = (df_ventas['Revenue'] / df_ventas['Revenue'].sum() * 100).round(1)

print("── Evolución mensual de ventas ──")
print(df_ventas.to_string(index=False))

print(f"\n── Estadísticas del ticket promedio mensual ──")
print(f"Media   : £{df_ventas['TicketPromedio'].mean():.2f}")
print(f"Mediana : £{df_ventas['TicketPromedio'].median():.2f}")
print(f"Desvío  : £{df_ventas['TicketPromedio'].std():.2f}")

C:\Users\Alexander\AppData\Local\Temp\ipykernel_11376\3991819282.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


── Evolución mensual de ventas ──
    Mes  Revenue  Pedidos  TicketPromedio  RevenuePct
2023-07 12885.93       21          613.62        26.1
2023-08 13837.25       20          691.86        28.1
2023-09  9599.50       21          457.12        19.5
2023-10 10112.40       22          459.65        20.5
2023-11  2869.40        6          478.23         5.8

── Estadísticas del ticket promedio mensual ──
Media   : £540.10
Mediana : £478.23
Desvío  : £106.80


---
## ✏️ Ejercicios — Parte 2

**Ejercicio 6.** Mostrá los 5 productos más vendidos (por cantidad total). Incluí nombre del producto, categoría y cantidad total vendida.

**Ejercicio 7.** ¿Qué proveedor genera más revenue en total? Calculá el revenue por proveedor, ordenado de mayor a menor.

**Ejercicio 8.** Calculá el tiempo promedio de envío (días entre `OrderDate` y `ShippedDate`) por empresa de envío (`Shippers`). ¿Cuál es más rápida?

**Ejercicio 9.** Encontrá los clientes que hicieron **más de 3 pedidos** y cuyo **revenue total supera £1000**. Mostrá cliente, país, cantidad de pedidos y revenue total.

**Ejercicio 10.** Usando la vista `vw_orden_detalle`, encontrá el empleado con mayor revenue en cada país de envío. *(Pista: subconsulta o CTE)*

In [28]:
# Ejercicio 6


In [29]:
# Ejercicio 7


In [30]:
# Ejercicio 8


In [31]:
# Ejercicio 9


In [32]:
# Ejercicio 10


---
## 🗂️ Referencia rápida

```sql
-- Estructura base
SELECT  col1, col2, funcion(col3) AS alias
FROM    TablaA a
JOIN    TablaB b  ON a.id = b.id
WHERE   condicion
GROUP BY col1, col2
HAVING  condicion_sobre_agregados
ORDER BY alias DESC
LIMIT   n;

-- Funciones de agregación
COUNT(*)          -- cuenta filas
COUNT(DISTINCT x) -- cuenta valores únicos
SUM(x)            -- suma
AVG(x)            -- promedio
MAX(x) / MIN(x)   -- máximo / mínimo

-- Funciones de fecha
DATE_FORMAT(fecha, '%Y-%m')   -- año-mes
DATEDIFF(fecha1, fecha2)      -- diferencia en días
MONTH(fecha) / YEAR(fecha)    -- mes / año

-- Tipos de JOIN
INNER JOIN  -- solo filas que coinciden en ambas tablas
LEFT JOIN   -- todas las filas de la tabla izquierda
RIGHT JOIN  -- todas las filas de la tabla derecha
```

In [33]:
# ── Cerrá la conexión cuando termines de trabajar ──
conn.close()
print('🔒 Conexión cerrada')

🔒 Conexión cerrada
